# 02 — Pipeline and experimental runs

Defines the four agents and runs every experimental condition: the three SKG conditions
and the three perceptual inputs. Also regenerates the captions with in-context examples and
verifies the explanations against the image.

Requires `skg.py` and `datos.py`. Writes `pipeline.py`.

**These runs take hours of GPU time.** On an A40, roughly 4–6 h for the single-frame
condition, 3–4 h without knowledge, 12–16 h for ten frames and 4–6 h for the oracle. Progress
is saved after each scenario, so an interruption loses nothing, but do not run the whole
notebook in one go.

## The pipeline module

Three decisions are worth flagging. Generation is greedy (`do_sample=False`), because Qwen
samples by default and without fixing this two runs of the same experiment give different
numbers. The visual token budget is capped because the subset mixes two resolutions. And the
catalogue shown to the hazard agent exposes only the conditions, never the actions, so that it
chooses by matching the scene rather than by reasoning backwards from a desirable manoeuvre.

In [ ]:
%%writefile ../src/pipeline.py
"""
pipeline.py — Los agentes y la orquestacion del pipeline multi-agente.

Arquitectura:
    Perception -> Hazard Reasoning -> [SKG] -> Causal Explanation -> Caption -> Validator
    y la rama de ablacion SIN SKG, que reutiliza la MISMA percepcion.

Requiere GPU (Qwen2.5-VL). Para analisis offline usar evaluacion.py.
"""

import os
import gc
import json
import time
from datetime import datetime

import torch
from transformers import Qwen2_5_VLForConditionalGeneration, AutoProcessor

from skg import SKG, LISTA_ACCIONES
from datos import frames_del_escenario, muestrear_frames

MODELO_ID = "Qwen/Qwen2.5-VL-7B-Instruct"

# Estado global del modelo (se carga una sola vez)
_model = None
_processor = None


# ===========================================================================
# 1. CARGA DEL MODELO
# ===========================================================================
def cargar_modelo(modelo_id=MODELO_ID, revision=None):
    """Carga el modelo y el procesador en GPU. Idempotente.

    revision: hash del commit en HuggingFace. Fijarlo hace la corrida
    reproducible aunque el repo cambie.
    """
    global _model, _processor
    if _model is not None:
        return _model, _processor

    kwargs = {"torch_dtype": torch.bfloat16, "device_map": "auto"}
    if revision:
        kwargs["revision"] = revision

    _model = Qwen2_5_VLForConditionalGeneration.from_pretrained(modelo_id, **kwargs)
    _processor = AutoProcessor.from_pretrained(modelo_id)
    print(f"Modelo cargado: {modelo_id} | GPU: {torch.cuda.get_device_name(0)}")
    return _model, _processor


def limpiar_gpu():
    """Libera memoria entre escenarios. gc primero, luego el cache de PyTorch."""
    gc.collect()
    torch.cuda.empty_cache()


# ===========================================================================
# 2. GENERACION (determinista)
# ===========================================================================
# do_sample=False => decodificacion greedy. Sin esto Qwen muestrea por defecto
# y dos corridas del mismo experimento dan numeros distintos.

def generar_texto(prompt, max_new_tokens=400):
    """Llamada solo-texto al modelo."""
    model, processor = cargar_modelo()
    messages = [{"role": "user", "content": [{"type": "text", "text": prompt}]}]
    text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = processor(text=[text], padding=True, return_tensors="pt").to(model.device)

    ids = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    # generate() devuelve prompt + respuesta: recortamos el prompt
    recortado = [o[len(i):] for i, o in zip(inputs.input_ids, ids)]
    return processor.batch_decode(recortado, skip_special_tokens=True)[0]

# Límite de tokens visuales por imagen. Necesario porque el pool mezcla
# resoluciones (2704x1520 y 1920x1208) y Qwen no redimensiona por defecto:
# una imagen de 2704x1520 genera ~5200 tokens visuales.
MAX_PIXELS = 1003520      # ≈1280 tokens/imagen

def generar_con_imagenes(rutas, prompt, max_new_tokens=600):
    """Llamada imagen(es)+texto. Acepta una ruta o una lista de rutas."""
    from qwen_vl_utils import process_vision_info

    model, processor = cargar_modelo()
    if isinstance(rutas, str):
        rutas = [rutas]

    contenido = [{"type": "image", "image": r, "max_pixels":MAX_PIXELS} for r in rutas]
    contenido.append({"type": "text", "text": prompt})
    messages = [{"role": "user", "content": contenido}]

    text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    imagenes, videos = process_vision_info(messages)
    inputs = processor(text=[text], images=imagenes, videos=videos,
                       padding=True, return_tensors="pt").to(model.device)

    ids = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    recortado = [o[len(i):] for i, o in zip(inputs.input_ids, ids)]
    return processor.batch_decode(recortado, skip_special_tokens=True,
                                  clean_up_tokenization_spaces=False)[0]


# ===========================================================================
# 3. PARSEO DE JSON (una sola version, antes habia tres)
# ===========================================================================
def limpiar_json(texto):
    """Extrae el bloque {...} de la respuesta del modelo.

    El modelo suele envolver el JSON en markdown o anteponer una frase.
    """
    texto = texto.strip()
    inicio, fin = texto.find("{"), texto.rfind("}")
    if inicio != -1 and fin != -1 and fin > inicio:
        return texto[inicio:fin + 1]
    return texto


def parsear(crudo):
    """Devuelve (dict, crudo) o (None, crudo) si el JSON no es valido."""
    try:
        return json.loads(limpiar_json(crudo)), crudo
    except json.JSONDecodeError:
        return None, crudo


# ===========================================================================
# 4. PROMPTS
# ===========================================================================
PROMPT_PERCEPTION = """You are a road-safety perception system analyzing a dashcam image from the ego-vehicle's perspective.

Return ONLY a valid JSON object with this exact structure, no extra text:
{
  "event": "<one sentence describing what is happening>",
  "scene": "<road type, weather, traffic density>",
  "traffic_light_state": "<red | amber | green | not_visible | no_traffic_light>",
  "objects": [
    {
      "label": "<object>",
      "location": "<where in the scene>",
      "distance": "<near | mid | far>",
      "influence_on_ego_path": "<direct | potential | none>",
      "relevant_to_safety": true
    }
  ],
  "initial_justification": "<why this scene might involve risk, if at all>"
}

Field rules:
- "traffic_light_state" refers ONLY to a light governing the EGO-VEHICLE's direction. If visible but unclear, use "not_visible".
- "influence_on_ego_path": "direct" = in or entering the ego trajectory NOW; "potential" = could enter soon; "none" = does not affect it.
- Describe ONLY what is visible. Return ONLY the JSON."""

PROMPT_PERCEPTION_TEMPORAL = """You are a road-safety perception system analyzing a SEQUENCE of dashcam frames from the ego-vehicle's perspective, ordered in time (earliest first, latest last).

By comparing the frames, infer MOTION and CHANGE over time.

Return ONLY a valid JSON object with this exact structure, no extra text:
{
  "event": "<what is happening across the sequence, including motion>",
  "scene": "<road type, weather, traffic density>",
  "traffic_light_state": "<red | amber | green | not_visible | no_traffic_light>",
  "objects": [
    {
      "label": "<object>",
      "location": "<where>",
      "motion": "<approaching | receding | stopped | crossing | stable>",
      "distance": "<near | mid | far>",
      "influence_on_ego_path": "<direct | potential | none>",
      "relevant_to_safety": true
    }
  ],
  "temporal_change": "<the key change between first and last frame>",
  "initial_justification": "<why this situation might involve risk>"
}

Field rules:
- "traffic_light_state" refers ONLY to a light governing the EGO-VEHICLE's direction. If visible but unclear, use "not_visible".
- "influence_on_ego_path": "direct" = in or entering the ego trajectory NOW; "potential" = could enter soon; "none" = does not affect it.
- "motion" describes how the object moves ACROSS THE FRAMES relative to the ego-vehicle, not its position in a single frame, must be EXACTLY ONE of: approaching, receding, stopped, crossing, stable. Copy the value verbatim; do not invent other values.
- Describe ONLY what is visible. Return ONLY the JSON."""

PROMPT_HAZARD = """You are a road-safety hazard reasoning system for the EGO-VEHICLE.

Given this structured scene perception:
{percepcion}

And this catalogue of known hazard types:
{catalogo}

Choose the ONE hazard that poses the most direct and immediate risk to the ego-vehicle's own path.

Return ONLY a JSON object:
{{"hazard_name": "<one name EXACTLY from the catalogue>", "evidence": "<what in the perception supports it>"}}"""

PROMPT_HAZARD_LIBRE = """You are a road-safety hazard reasoning system for the EGO-VEHICLE.

Given this structured scene perception:
{percepcion}

Identify the ONE hazard that poses the most direct and immediate risk to the ego-vehicle's own path.

Return ONLY a JSON object:
{{"hazard_name": "<a short name for the hazard, in your own words>", "evidence": "<what in the perception supports it>"}}"""

PROMPT_CAUSAL = """You are a road-safety explanation system for the EGO-VEHICLE (the car whose dashcam recorded the scene).

Scene perception:
{percepcion}

Identified hazard and retrieved safety knowledge:
{conocimiento}

Return ONLY a JSON object:
{{
  "hazard": "<the hazard, one clear sentence>",
  "evidence": "<what in the scene supports it>",
  "causal_chain": "<step by step: why this situation creates risk FOR THE EGO-VEHICLE>",
  "rule_violated": "<the traffic rule involved, in plain language>",
  "recommended_action": "<EXACTLY ONE label from the list below>"
}}

STRICT RULES:
1. "recommended_action" must be EXACTLY ONE of these labels, copied verbatim, with no additions and no "or":
{acciones}
2. The knowledge may describe conditions for different actions. You must RESOLVE the condition using the perception and output only the single action that applies to THIS scene.
3. The causal chain must explain risk TO THE EGO-VEHICLE, and the action is what THE EGO-DRIVER must do. Never reason about risks between other road users unless they directly affect the ego path.
4. Regulatory signals governing the ego path (red traffic lights, stop signs, stopped lead vehicles) take precedence over agents that are NOT in the ego path.
5. Use ONLY the perception and the knowledge provided. Do not invent facts."""

PROMPT_SIN_SKG = """You are a road-safety system for the EGO-VEHICLE. Given this dashcam scene perception:
{percepcion}

What should the ego-driver do? Return ONLY a JSON object:
{{"recommended_action": "<EXACTLY ONE label from the list below, copied verbatim>", "justification": "<why>"}}

The action must be EXACTLY ONE of:
{acciones}"""

PROMPT_CAPTION = """You are a road-safety captioning system for the EGO-VEHICLE.

Scene perception:
{percepcion}

Causal explanation:
{explicacion}

Examples of the target caption style:
- "There is a black sedan accelerating in front of the ego car, in the ego lane, because traffic has started moving."
- "There is a crossing guard standing to the left of the road, because he is directing pedestrian traffic."
- "There is construction work blocking the left lane, ego car must stop and wait until given the signal to proceed."
- "There is a white motorcycle slowing in front of the ego car, in the ego lane, because of traffic congestion ahead."
- "There is a cyclist wearing brown shirt in front of the ego car cycling towards the ego car."

Write ONE caption following the style of the examples: begin with "There is", name the key risk object with its visual attributes, state what it is doing and where it is relative to the ego car, and give the reason with "because" when the perception supports one. Aim for 18-22 words.

Return ONLY a JSON object:
{{"caption": "<the single sentence>"}}"""

PROMPT_VALIDATOR = """You are a validation system checking a road-safety explanation for HALLUCINATIONS and INCONSISTENCIES.

Scene perception:
{percepcion}

Retrieved safety knowledge (the rule and prescribed action for the identified hazard):
{conocimiento}

Explanation to validate:
{explicacion}

Caption to validate:
{caption}

The perception and the retrieved knowledge together constitute the ONLY grounds available. A claim supported by either of them is valid.

Return ONLY a JSON object:
{{
  "verdict": "<pass | fail>",
  "hallucinated_claims": ["<claims asserting objects or states absent from BOTH the perception and the retrieved knowledge>"],
  "inconsistencies": ["<contradictions between explanation, caption, perception and knowledge>"],
  "ego_frame_ok": "<true | false>"
}}

Rules:
- A claim is hallucinated only if it asserts something absent from, or contradicted by, both the perception and the retrieved knowledge. Reformulations of either source are valid.
- Statements of a traffic rule or of a prescribed action are valid when they derive from the retrieved knowledge, even if they do not appear in the perception.
- "ego_frame_ok" is false if the explanation drifts into reasoning about risks BETWEEN OTHER road users instead of risk to the ego-vehicle.
- Verdict is "pass" only if there are no hallucinations, no inconsistencies and ego_frame_ok is true.
- Return empty lists if nothing is found."""


# ===========================================================================
# 5. LOS AGENTES
# ===========================================================================
def agente_percepcion(ruta_imagen):
    """Frame unico -> escena estructurada."""
    return parsear(generar_con_imagenes(ruta_imagen, PROMPT_PERCEPTION, 800))


def agente_percepcion_temporal(ruta_frame_anotado, n_frames=3):
    """Secuencia de frames -> escena estructurada con informacion de movimiento."""
    frames = frames_del_escenario(ruta_frame_anotado)
    seleccion = muestrear_frames(frames, n_frames, frame_anotado=ruta_frame_anotado)
    return parsear(generar_con_imagenes(seleccion, PROMPT_PERCEPTION_TEMPORAL, 900))


def agente_hazard(percepcion):
    """Elige un peligro del catalogo y recupera su subgrafo del SKG.

    El catalogo expone solo las CONDICIONES, no las acciones: asi el agente
    elige por similitud con la escena y no razona hacia atras desde la accion.
    """
    catalogo = "\n".join(f"- {k}: {v['condition']}" for k, v in SKG.items())
    prompt = PROMPT_HAZARD.format(percepcion=json.dumps(percepcion, indent=2),
                                  catalogo=catalogo)
    eleccion, crudo = parsear(generar_texto(prompt, 200))
    if eleccion is None:
        return None, crudo

    nombre = eleccion.get("hazard_name", "")
    return {
        "hazard_name": nombre,
        "evidence": eleccion.get("evidence", ""),
        "subgraph": SKG.get(nombre),  # None si el modelo invento un nombre
    }, crudo


def agente_causal(percepcion, razonamiento):
    """Causal explanation, with the action resolved to a single enum label."""
    if not razonamiento or not razonamiento.get("subgraph"):
        return None, "subgrafo ausente"
    conocimiento = {
        "hazard_name": razonamiento["hazard_name"],
        "evidence": razonamiento["evidence"],
        **razonamiento["subgraph"],
    }
    prompt = PROMPT_CAUSAL.format(percepcion=json.dumps(percepcion, indent=2),
                                  conocimiento=json.dumps(conocimiento, indent=2),
                                  acciones=LISTA_ACCIONES)
    return parsear(generar_texto(prompt, 400))


def agente_caption(percepcion, explicacion):
    """Caption final estilo DRAMA."""
    prompt = PROMPT_CAPTION.format(percepcion=json.dumps(percepcion, indent=2),
                                   explicacion=json.dumps(explicacion, indent=2))
    return parsear(generar_texto(prompt, 120))


def agente_validador(percepcion, explicacion, caption, conocimiento=None):
    """Comprueba alucinaciones, inconsistencias y perdida del marco egocentrico.

    Recibe tambien el subgrafo recuperado: sin el, penaliza afirmaciones
    correctamente fundamentadas en el conocimiento normativo.
    Limitacion declarada: valida contra la percepcion, no contra la imagen,
    de modo que no detecta errores originados en la percepcion misma.
    """
    prompt = PROMPT_VALIDATOR.format(
        percepcion=json.dumps(percepcion, indent=2),
        conocimiento=json.dumps(conocimiento or {}, indent=2),
        explicacion=json.dumps(explicacion, indent=2),
        caption=json.dumps(caption or {}, indent=2))
    return parsear(generar_texto(prompt, 300))

def rama_sin_skg(percepcion):
    """Baseline de la ablacion: percepcion -> accion, sin grafo ni razonamiento.

    Recibe la percepcion YA calculada para que ambas ramas partan del mismo
    input. Eso es lo que hace que la comparacion aisle la contribucion del SKG.
    """
    prompt = PROMPT_SIN_SKG.format(percepcion=json.dumps(percepcion, indent=2),
                                   acciones=LISTA_ACCIONES)
    return parsear(generar_texto(prompt, 200))

def agente_hazard_libre(percepcion):
    prompt = PROMPT_HAZARD_LIBRE.format(percepcion=json.dumps(percepcion, indent=2))
    eleccion, crudo = parsear(generar_texto(prompt, 200))
    if eleccion is None:
        return None, crudo
    return {"hazard_name": eleccion.get("hazard_name", ""),
            "evidence": eleccion.get("evidence", "")}, crudo

def agente_causal_libre(percepcion, razonamiento):
    """Mismo prompt causal, pero el 'conocimiento' contiene solo lo que el
    propio modelo identifico: ni catalogo, ni regla, ni accion prescrita."""
    conocimiento = {"hazard_name": razonamiento["hazard_name"],
                    "evidence": razonamiento["evidence"]}
    prompt = PROMPT_CAUSAL.format(percepcion=json.dumps(percepcion, indent=2),
                                  conocimiento=json.dumps(conocimiento, indent=2),
                                  acciones=LISTA_ACCIONES)
    return parsear(generar_texto(prompt, 400))


# ===========================================================================
# 6. ORQUESTACION
# ===========================================================================
def pipeline_con_skg(ruta_imagen, temporal=False, n_frames=1):
    """Ejecuta los agentes en cadena. Devuelve todo, incluido donde fallo."""
    salida = {"percepcion": None, "razonamiento": None, "explicacion": None,
              "caption": None, "validacion": None, "fallo": None,
              "n_frames": n_frames if temporal else 1}

    if temporal:
        percepcion, crudo = agente_percepcion_temporal(ruta_imagen, n_frames)
    else:
        percepcion, crudo = agente_percepcion(ruta_imagen)
    salida["percepcion"] = percepcion
    if percepcion is None:
        salida["fallo"] = "parse_percepcion"
        salida["crudo"] = crudo[:300]
        return salida

    razonamiento, crudo = agente_hazard(percepcion)
    salida["razonamiento"] = razonamiento
    if razonamiento is None:
        salida["fallo"] = "parse_hazard"
        salida["crudo"] = crudo[:300]
        return salida
    if razonamiento["subgraph"] is None:
        salida["fallo"] = "hazard_fuera_de_catalogo"
        return salida

    explicacion, crudo = agente_causal(percepcion, razonamiento)
    salida["explicacion"] = explicacion
    if explicacion is None:
        salida["fallo"] = "parse_causal"
        salida["crudo"] = crudo[:300]
        return salida

    caption, _ = agente_caption(percepcion, explicacion)
    salida["caption"] = caption

    conocimiento = {**razonamiento["subgraph"], "hazard_name": razonamiento["hazard_name"]}
    validacion, _ = agente_validador(percepcion, explicacion, caption, conocimiento)
    salida["validacion"] = validacion
    return salida


def pipeline_sin_conocimiento(ruta_imagen):
    """Condicion B: mismas etapas de razonamiento, sin catalogo ni subgrafo."""
    salida = {"percepcion": None, "razonamiento": None, "explicacion": None, "fallo": None}
    percepcion, crudo = agente_percepcion(ruta_imagen)
    salida["percepcion"] = percepcion
    if percepcion is None:
        salida["fallo"] = "parse_percepcion"; return salida
    rz, crudo = agente_hazard_libre(percepcion)
    salida["razonamiento"] = rz
    if rz is None:
        salida["fallo"] = "parse_hazard"; return salida
    exp, crudo = agente_causal_libre(percepcion, rz)
    salida["explicacion"] = exp
    if exp is None:
        salida["fallo"] = "parse_causal"
    return salida

def procesar_escenario(esc, config, temporal=False, n_frames=1):
    """Un escenario completo: rama CON SKG + rama SIN SKG, listo para guardar."""
    con = pipeline_con_skg(esc["ruta"], temporal=temporal, n_frames=n_frames)

    resultado = {
        "indice": esc["indice"],
        "tipo": esc["tipo"],
        "caption_gt": esc["caption"],
        "sugerencia_gt": esc["sugerencia"],
        "config": config,
        "percepcion": con["percepcion"],   # se guarda para inspeccionar sin re-ejecutar
        "con_skg": {
            "hazard": (con["razonamiento"] or {}).get("hazard_name"),
            "accion_cruda": (con["explicacion"] or {}).get("recommended_action", ""),
            "explicacion": con["explicacion"],
            "caption": (con["caption"] or {}).get("caption", ""),
            "validacion": con["validacion"],
            "fallo": con["fallo"],
        },
    }

    # La rama sin SKG solo tiene sentido si hubo percepcion
    if con["percepcion"] is not None:
        sin, _ = rama_sin_skg(con["percepcion"])
        resultado["sin_skg"] = {
            "accion_cruda": (sin or {}).get("recommended_action", ""),
            "fallo": None if sin else "parse_sin_skg",
        }
    else:
        resultado["sin_skg"] = {"accion_cruda": "", "fallo": "sin_percepcion"}

    return resultado


# ===========================================================================
# 7. BUCLE DE CORRIDA (reanudable y con presupuesto de tiempo)
# ===========================================================================
def correr_lote(lote, ruta_salida, config, temporal=False, n_frames=1,
                presupuesto_min=None):
    """Procesa el lote guardando tras cada escenario.

    - Si ruta_salida ya existe, reanuda: se salta los indices ya hechos.
    - Si presupuesto_min esta puesto, para limpiamente antes de agotarlo
      (nunca a mitad de una inferencia).
    """
    if os.path.exists(ruta_salida):
        with open(ruta_salida) as f:
            resultados = json.load(f)
        hechos = {r["indice"] for r in resultados}
        print(f"Reanudando: {len(hechos)} escenarios ya hechos")
    else:
        os.makedirs(os.path.dirname(ruta_salida), exist_ok=True)
        resultados, hechos = [], set()

    inicio = time.monotonic()
    duraciones = []

    for k, esc in enumerate(lote, 1):
        if esc["indice"] in hechos:
            continue

        # Control de presupuesto: estimamos con el promedio observado + 20%
        if presupuesto_min is not None:
            restante = presupuesto_min * 60 - (time.monotonic() - inicio)
            estimado = (sum(duraciones) / len(duraciones)) * 1.2 if duraciones else 180
            if restante < estimado:
                print(f"\nPresupuesto agotado ({restante:.0f}s restantes). Paro limpio.")
                break

        t0 = time.monotonic()
        print(f"[{k}/{len(lote)}] {esc['tipo']} idx {esc['indice']}", end=" ", flush=True)

        resultado = procesar_escenario(esc, config, temporal=temporal, n_frames=n_frames)
        resultados.append(resultado)
        with open(ruta_salida, "w") as f:
            json.dump(resultados, f, indent=2, ensure_ascii=False)

        duraciones.append(time.monotonic() - t0)
        fallo = resultado["con_skg"]["fallo"]
        print(f"| CON: {resultado['con_skg']['accion_cruda'][:24]:<24} "
              f"| SIN: {resultado['sin_skg']['accion_cruda'][:24]:<24} "
              f"| {duraciones[-1]:.0f}s"
              + (f" | FALLO: {fallo}" if fallo else ""))
        limpiar_gpu()

    print(f"\nGuardados {len(resultados)} escenarios en {ruta_salida}")
    return resultados


def nueva_config(**extra):
    """Metadatos que se guardan dentro de cada resultado, para trazabilidad."""
    config = {
        "modelo": MODELO_ID,
        "decoding": "greedy (do_sample=False)",
        "percepcion": "v3_structured",
        "accion": "enum_cerrado",
        "max_pixels": MAX_PIXELS,
        "fecha": datetime.now().isoformat(timespec="seconds"),
    }
    config.update(extra)
    return config

## Temporal prompt versions

The temporal arm was run under three successive prompt specifications, to rule out that the
degradation came from a poor field definition rather than from the temporal context itself.
The first referred back to the single-frame rules instead of restating them, and since each
agent call is independent the model never saw them. The second spelled them out. The third
added the closed set on the motion field.

Only the third produced the reported results; the other two are kept so the check can be
reproduced. Schema compliance went from 85% under V2 to 94% under V3, and exact match was
111, 105 and 101 against 132 single-frame: the degradation held under all three.

In [ ]:
PROMPT_TEMPORAL_V1_TAIL = """Same field rules as single-frame perception. Return ONLY the JSON."""

PROMPT_TEMPORAL_V2_TAIL = """Field rules:
- "traffic_light_state" refers ONLY to a light governing the EGO-VEHICLE's direction. If visible but unclear, use "not_visible".
- "influence_on_ego_path": "direct" = in or entering the ego trajectory NOW; "potential" = could enter soon; "none" = does not affect it.
- Describe ONLY what is visible. Return ONLY the JSON."""

# V3 is the one in pipeline.py
PROMPT_TEMPORAL_V3_TAIL = """Field rules:
- "traffic_light_state" refers ONLY to a light governing the EGO-VEHICLE's direction. If visible but unclear, use "not_visible".
- "influence_on_ego_path": "direct" = in or entering the ego trajectory NOW; "potential" = could enter soon; "none" = does not affect it.
- "motion" describes how the object moves ACROSS THE FRAMES relative to the ego-vehicle, must be EXACTLY ONE of: approaching, receding, stopped, crossing, stable. Copy the value verbatim.
- Describe ONLY what is visible. Return ONLY the JSON."""

## Setup

The revision hash pins the exact weights. Without it, `from_pretrained` takes whatever is on
the repository's main branch, which can change.

In [ ]:
import sys, os, json, re, random, time
from collections import Counter, defaultdict

sys.path.insert(0, "../src")   # project modules
import torch, transformers

from datos import cargar_escenarios
from pipeline import (cargar_modelo, correr_lote, nueva_config, limpiar_gpu,
                      pipeline_con_skg, pipeline_sin_conocimiento, rama_sin_skg, MAX_PIXELS)
from skg import SKG, CLASE_DE, LISTA_ACCIONES, normalizar_accion

RESULTADOS = os.environ.get("RESULTS_DIR", "../results")
REVISION = "cc594898137f460bfe9f0759e9844b3ce807cfb5"

escenarios = cargar_escenarios()
escenarios.sort(key=lambda e: e["indice"])
print(f"Full pool on disk: {len(escenarios)}")

# The evaluated subset is the 520 balanced scenarios, not the whole pool.
indices_520 = {r["indice"] for r in json.load(open(f"{RESULTADOS}/pool520_1f.json"))}
por_idx = {e["indice"]: e for e in escenarios}
lote_520 = [por_idx[i] for i in sorted(indices_520) if i in por_idx]
print(f"Evaluation subset: {len(lote_520)}", dict(Counter(e["tipo"] for e in lote_520)))
print(f"transformers {transformers.__version__} | torch {torch.__version__}")

A stale kernel holding the model makes `device_map="auto"` spill layers onto CPU, and
inference goes from thirty seconds to several minutes per scenario without raising anything.

In [ ]:
!nvidia-smi --query-gpu=memory.used,memory.total,utilization.gpu --format=csv

modelo, _ = cargar_modelo(revision=REVISION)
print("device:", next(modelo.parameters()).device)
print(f"VRAM in use: {torch.cuda.memory_allocated()/1e9:.1f} GB")

## Conditions A and C — single frame

The main run. Each scenario goes through the full pipeline with the graph (A) and, over the
**same perception**, through the direct inference that serves as baseline (C). Both branches
starting from the already-computed perception is what makes the comparison valid: the only
thing that varies is the reasoning that follows.

In [ ]:
correr_lote(lote_520,
            f"{RESULTADOS}/pool520_1f.json",
            nueva_config(experimento="single_frame", skg="graduado_v2",
                         transformers=transformers.__version__,
                         torch=torch.__version__, model_revision=REVISION))

## Condition B — same stages, no knowledge

Keeps hazard identification and causal explanation, but the agent names the hazard in its own
words with no catalogue, and reasons with no subgraph. Contrasting A against B isolates the
contribution of the normative knowledge; contrasting B against C isolates the scaffolding.

In [ ]:
RUTA_B = f"{RESULTADOS}/pool520_sin_conocimiento.json"
res = json.load(open(RUTA_B)) if os.path.exists(RUTA_B) else []
hechos = {r["indice"] for r in res}

cfg_B = nueva_config(experimento="condicion_B_sin_conocimiento", skg="ninguno",
                     transformers=transformers.__version__, torch=torch.__version__,
                     model_revision=REVISION,
                     nota="same reasoning stages, no catalogue and no subgraph")

for k, esc in enumerate(lote_520, 1):
    if esc["indice"] in hechos:
        continue
    t0 = time.monotonic()
    print(f"[{k}/{len(lote_520)}] {esc['tipo']} idx {esc['indice']}", end=" ", flush=True)

    out = pipeline_sin_conocimiento(esc["ruta"])
    res.append({
        "indice": esc["indice"], "tipo": esc["tipo"], "config": cfg_B,
        "caption_gt": esc["caption"], "sugerencia_gt": esc["sugerencia"],
        "percepcion": out["percepcion"],
        # Same field name as condition A so evaluacion.py reads it unchanged
        "con_skg": {"hazard": (out["razonamiento"] or {}).get("hazard_name"),
                    "accion_cruda": (out["explicacion"] or {}).get("recommended_action", ""),
                    "explicacion": out["explicacion"], "fallo": out["fallo"]},
        "sin_skg": {"accion_cruda": "", "fallo": "no_aplica"},
    })
    json.dump(res, open(RUTA_B, "w"), indent=2, ensure_ascii=False)
    print(f"| {res[-1]['con_skg']['accion_cruda'][:24]} | {time.monotonic()-t0:.0f}s")
    limpiar_gpu()

print(f"\nDone: {len(res)}")

## Temporal context — ten frames

Same configuration, changing only the perception agent's input. Frames are sampled evenly
across the clip and always include the annotated one, which is what the reference annotation
refers to. This is the longest run of the project.

In [ ]:
correr_lote(lote_520,
            f"{RESULTADOS}/pool520_10f.json",
            nueva_config(experimento="temporal_10f", skg="graduado_v2",
                         temporal=True, n_frames=10,
                         percepcion="v3_temporal_motion_enum",
                         transformers=transformers.__version__,
                         torch=torch.__version__, model_revision=REVISION),
            temporal=True, n_frames=10)

## Oracle localisation

The annotated box is injected into the perception prompt, normalised to a 0–1000 scale so it
does not depend on resolution. The hint says **where** the object is, not what it is or whether
it is dangerous. This is not a deployable configuration, since the annotation is not available
at run time; its purpose is to set an upper bound on what perfect localisation could add.

In [ ]:
from PIL import Image
import pipeline
from pipeline import PROMPT_PERCEPTION

from datos import RUTA_ANOTACIONES
data = json.load(open(RUTA_ANOTACIONES))


def bbox_de(anotacion):
    """Axis-aligned box from the annotated polygon, or None if degenerate."""
    pts = anotacion.get("geometry")
    if not pts or len(pts) < 3:
        return None
    xs, ys = [p[0] for p in pts], [p[1] for p in pts]
    if max(xs) - min(xs) <= 0 or max(ys) - min(ys) <= 0:
        return None
    return [min(xs), min(ys), max(xs), max(ys)]


def pista_localizacion(ruta, bbox):
    with Image.open(ruta) as im:
        w, h = im.size
    n = [round(bbox[0]*1000/w), round(bbox[1]*1000/h),
         round(bbox[2]*1000/w), round(bbox[3]*1000/h)]
    return (f"LOCALISATION HINT (ground-truth): the safety-relevant object lies within "
            f"the region [{n[0]}, {n[1]}, {n[2]}, {n[3]}] on a normalised 0-1000 scale "
            f"(origin top-left). This indicates WHERE it is, not what it is or whether "
            f"it is dangerous. Identify and describe it yourself, and continue to report "
            f"every other object you can see.\n\n")


_pista_actual = {}

def percepcion_con_oraculo(ruta_imagen):
    marcador = "Return ONLY a valid JSON object"
    prompt = PROMPT_PERCEPTION.replace(marcador, _pista_actual["texto"] + marcador, 1)
    return pipeline.parsear(pipeline.generar_con_imagenes(ruta_imagen, prompt, 800))

_percepcion_original = pipeline.agente_percepcion
pipeline.agente_percepcion = percepcion_con_oraculo
print("Perception agent swapped for the oracle variant.")

In [ ]:
RUTA_O = f"{RESULTADOS}/pool520_bbox.json"
res_o = json.load(open(RUTA_O)) if os.path.exists(RUTA_O) else []
hechos_o = {r["indice"] for r in res_o}

cfg_O = nueva_config(experimento="oraculo_bbox", skg="graduado_v2",
                     percepcion="v3_bbox_oracle",
                     transformers=transformers.__version__, torch=torch.__version__,
                     model_revision=REVISION)

con_caja = [e for e in lote_520 if bbox_de(data[e["indice"]])]
print(f"Scenarios with a usable box: {len(con_caja)}/{len(lote_520)}")

for k, esc in enumerate(con_caja, 1):
    if esc["indice"] in hechos_o:
        continue
    _pista_actual["texto"] = pista_localizacion(esc["ruta"], bbox_de(data[esc["indice"]]))
    t0 = time.monotonic()
    print(f"[{k}/{len(con_caja)}] idx {esc['indice']}", end=" ", flush=True)

    con = pipeline_con_skg(esc["ruta"])
    r = {"indice": esc["indice"], "tipo": esc["tipo"], "config": cfg_O,
         "caption_gt": esc["caption"], "sugerencia_gt": esc["sugerencia"],
         "percepcion": con["percepcion"],
         "con_skg": {"hazard": (con["razonamiento"] or {}).get("hazard_name"),
                     "accion_cruda": (con["explicacion"] or {}).get("recommended_action", ""),
                     "explicacion": con["explicacion"],
                     "caption": (con["caption"] or {}).get("caption", ""),
                     "validacion": con["validacion"], "fallo": con["fallo"]}}
    if con["percepcion"] is not None:
        sin, _ = rama_sin_skg(con["percepcion"])
        r["sin_skg"] = {"accion_cruda": (sin or {}).get("recommended_action", ""),
                        "fallo": None if sin else "parse_sin_skg"}
    else:
        r["sin_skg"] = {"accion_cruda": "", "fallo": "sin_percepcion"}

    res_o.append(r)
    json.dump(res_o, open(RUTA_O, "w"), indent=2, ensure_ascii=False)
    print(f"| {r['con_skg']['accion_cruda'][:24]} | {time.monotonic()-t0:.0f}s")
    limpiar_gpu()

pipeline.agente_percepcion = _percepcion_original
print(f"\nDone: {len(res_o)}. Perception agent restored.")

## Captions with in-context examples

The caption is a separate call over the already-saved perception and explanation, so it can be
regenerated without re-running the pipeline. The first version had no examples and produced
correct descriptions in a style quite unlike the annotations. Adding five examples taken from
scenarios **outside the evaluation subset** raised the lexical overlap metrics substantially
while semantic similarity barely moved.

In [ ]:
from pipeline import PROMPT_CAPTION, generar_texto, parsear

RUTA_CAP = f"{RESULTADOS}/pool520_captions_fewshot.json"
datos_cap = json.load(open(f"{RESULTADOS}/pool520_1f.json"))

res_cap = json.load(open(RUTA_CAP)) if os.path.exists(RUTA_CAP) else []
hechos_cap = {r["indice"] for r in res_cap}

for k, r in enumerate(datos_cap, 1):
    if r["indice"] in hechos_cap:
        continue
    p, exp = r.get("percepcion"), r["con_skg"].get("explicacion")
    if not p or not exp:
        res_cap.append({"indice": r["indice"], "caption": ""})
        continue
    prompt = PROMPT_CAPTION.format(percepcion=json.dumps(p, indent=2),
                                   explicacion=json.dumps(exp, indent=2))
    cap, _ = parsear(generar_texto(prompt, 150))
    res_cap.append({"indice": r["indice"], "caption": (cap or {}).get("caption", "")})
    if k % 50 == 0:
        json.dump(res_cap, open(RUTA_CAP, "w"), indent=2, ensure_ascii=False)
        print(f"[{k}/{len(datos_cap)}]", flush=True)
    limpiar_gpu()

json.dump(res_cap, open(RUTA_CAP, "w"), indent=2, ensure_ascii=False)
print(f"Done: {len(res_cap)}")

## Verifying the explanations against the image

The validator checks the explanation against the perception, not against the image, so it
cannot detect an error that originated in the perception itself. This check does look at the
image.

An independent seven-billion-parameter model (LLaVA-1.6-Mistral) was tried first and assigned
the maximum score on all three criteria to the entire subsample without discriminating, so the
pipeline's own model was used instead. That makes this self-evaluation, and the figures should
be read as an upper bound.

In [ ]:
from pipeline import generar_con_imagenes

PROMPT_JUEZ = """You are verifying an automatically generated safety explanation against the dashcam image shown.

Explanation:
{explicacion}

Recommended action for the ego-vehicle: {accion}

Examine the image and answer each question with one short sentence of justification followed by a score.

1. grounding: Are ALL the objects and states mentioned actually visible in the image? Score 1 if several are absent or contradicted, 2 if some are questionable, 3 if all are verifiable.
2. causal_validity: Does the explanation correctly account for why this situation poses a risk to the ego-vehicle? Score 1 if incoherent, 2 if partially sound, 3 if sound.
3. action_support: Does the recommended action follow from the explanation? Score 1 if unrelated, 2 if loosely related, 3 if it follows directly.

Be critical: not every explanation deserves a high score.

Answer exactly in this format:
grounding: <sentence> SCORE: <1 or 2 or 3>
causal_validity: <sentence> SCORE: <1 or 2 or 3>
action_support: <sentence> SCORE: <1 or 2 or 3>"""


def parsear_juicio(texto):
    d = {}
    for criterio in ["grounding", "causal_validity", "action_support"]:
        m = re.search(rf"{criterio}\s*:.*?SCORE\s*:\s*([123])", texto, re.I | re.S)
        if m:
            d[criterio] = int(m.group(1))
    return d if len(d) == 3 else None


def juzgar(ruta_imagen, explicacion, accion):
    prompt = PROMPT_JUEZ.format(
        explicacion=json.dumps({k: explicacion.get(k) for k in
                                ["hazard", "evidence", "causal_chain", "rule_violated"]},
                               indent=2),
        accion=accion)
    return parsear_juicio(generar_con_imagenes(ruta_imagen, prompt, 300))

In [ ]:
RUTA_J = f"{RESULTADOS}/juicio_qwen_520.json"
todos = json.load(open(f"{RESULTADOS}/pool520_1f.json"))
por_idx_esc = {e["indice"]: e for e in lote_520}

res_j = json.load(open(RUTA_J)) if os.path.exists(RUTA_J) else []
hechos_j = {r["indice"] for r in res_j}

for k, r in enumerate(todos, 1):
    if r["indice"] in hechos_j or not r["con_skg"].get("explicacion"):
        continue
    j = juzgar(por_idx_esc[r["indice"]]["ruta"],
               r["con_skg"]["explicacion"], r["con_skg"]["accion_cruda"])
    res_j.append({"indice": r["indice"], "tipo": r["tipo"], "juicio": j})
    if k % 25 == 0:
        json.dump(res_j, open(RUTA_J, "w"), indent=2, ensure_ascii=False)
        print(f"[{k}/{len(todos)}]", flush=True)
    limpiar_gpu()

json.dump(res_j, open(RUTA_J, "w"), indent=2, ensure_ascii=False)
validos = [x for x in res_j if x.get("juicio")]
print(f"\nValid judgements: {len(validos)}/{len(res_j)}")
for crit in ["grounding", "causal_validity", "action_support"]:
    v = [x["juicio"][crit] for x in validos]
    print(f"  {crit:<18} {sum(v)/len(v):.2f}/3  {dict(sorted(Counter(v).items()))}")

---

Once the runs are done, go back to notebook 01 and run the consolidation cell.